# Writing a quantum program 1 — register wiring

Compose an eight-bit signed register transformation from the shared complement and constant-add components. FlagQuantum owns the circuit IR, value codecs and drawing. Users append operations to a circuit. The signed word uses MSB-first two's-complement encoding.


## Declare the register

A Core signature declares the logical input/output word. An empty circuit preserves its eight input wires. The signature is metadata, while CircuitIR is the execution representation. Core verifies the signature and the physical IR validates the circuit.


In [ ]:
import flagquantum as fq
from flagquantum.core.ir.program.register_types import signed_register_type
from flagquantum.core.ir.program.registers import Register
from flagquantum.core.ir.program.signatures import Signature

width = 8
signature = Signature((Register("x", signed_register_type(width)),))
do_nothing = fq.Circuit(width)


Validate the identity circuit before execution. This checks structural consistency.


In [ ]:
do_nothing.to_ir().validate()
assert tuple((r.name, r.bitsize) for r in signature) == (("x", 8),)


Draw the input-to-output dependency using the existing register layout and Drawer.


In [ ]:
from importlib import import_module

from flagquantum.drawer.call_graph import render_call_graph_svg
from flagquantum.drawer.register_graph import export_register_graph_dot
from flagquantum.drawer.register_score import RegisterStep, layout_register_score

ipython_display = import_module("IPython.display")

identity_score = layout_register_score({"x": width}, ("x",), (), outputs=("x",))
identity_dot = export_register_graph_dot(identity_score)
ipython_display.display(ipython_display.SVG(render_call_graph_svg(identity_dot)))


## Signed integer encoding

Finite signed integers use two's-complement bits. The shared numerical codec encodes both 1 and -1 in eight bits.


In [ ]:
from flagquantum.simulation.integer_encoding import integer_to_bits

print(integer_to_bits(1, width, signed=True))
print(integer_to_bits(-1, width, signed=True))


For a fixed-width stored word, unary negation is `-x = ~x + 1` modulo `2**width`. This also defines the minimum signed value wrapping to itself. Complement flips the bits, and constant addition increments the result.


## Compose negation


The compiler components accept explicit MSB-first target wires. Complement requires no work. AddK(1) needs eight zero constant-load wires, seven zero carry wires and seven classical measurement slots. The caller supplies these separately from the eight logical input/output wires. Both appended operations share the same target. The addition restores its quantum workspace to zero in every supported measurement branch.


In [ ]:
from collections import Counter
from dataclasses import dataclass

from flagquantum.compiler.registers.arithmetic.addition import add_constant_ir
from flagquantum.compiler.registers.bits.bitwise import bitwise_not_ir
from flagquantum.core.ir import CircuitIR, Instruction


@dataclass(frozen=True)
class RegisterOperation:
    instructions: tuple[Instruction, ...]

    def append_to(self, circuit: fq.Circuit) -> fq.Circuit:
        return circuit.compose(CircuitIR(circuit.n_qubits, self.instructions))


target = tuple(range(width))
constant = tuple(range(width, 2 * width))
carries = tuple(range(2 * width, 3 * width - 1))
classical_bits = tuple(range(width - 1))
complement = bitwise_not_ir(target)
increment = add_constant_ir(target, 1, constant, carries, classical_bits)
negate = fq.Circuit(3 * width - 1)
negate.append(RegisterOperation(complement))
negate.append(RegisterOperation(increment))


Validate the emitted IR. The tutorial retains the original complement-then-increment sequence, including measured carry cleanup.


In [ ]:
negate.to_ir().validate()
assert negate.to_ir().instructions == complement + increment


Draw the two component calls with the same logical x dependency. Workspace is shown separately in the resource report. This is a register dependency graph, while the emitted CircuitIR retains every primitive and feedback instruction.


In [ ]:
from flagquantum.compiler._component_trace.resource_model import (
    estimate_component_resources,
)
from flagquantum.compiler.registers.arithmetic.constant_add_resource_model import (
    source_constant_add_cost,
)

steps = (
    RegisterStep("BitwiseNot", ("x",), ("x",)),
    RegisterStep("AddK(1)", ("x",), ("x",)),
)
negation_score = layout_register_score({"x": width}, ("x",), steps, outputs=("x",))
negation_dot = export_register_graph_dot(negation_score)
ipython_display.display(ipython_display.SVG(render_call_graph_svg(negation_dot)))
primitive_counts = dict(Counter(g.name for g in negate.to_ir().instructions))
print("Emitted primitive counts", primitive_counts)

model = estimate_component_resources(source_constant_add_cost(width, 1))
print("Independent AddK model", model)
print("Logical target width", width, "clean quantum work", len(constant) + len(carries))


## Test classical values

The shared classical model composes the same complement and increment value functions. Calling it with x=5 returns -5. It supports exhaustive checks of the finite signed domain without allocating the quantum state. This value check is separate from quantum simulation. The acceptance suite reuses exact primitive, complex-amplitude and cleanup evidence and enumerates every measurement branch at a small width.


In [ ]:
from flagquantum.simulation.classical_negation import negate_classical_value

negated_value = negate_classical_value(5, width, signed=True)
assert negated_value == -5
print({"x": negated_value})


## Provenance and execution scope

Adapted from the complete Qualtran v0.7.0 tutorial at revision `bcb17bb25360ed68ce3bc084645db3e929799389`, blob `061fa74a7a2a4213bd406ba9fdd16cbad07276c2`. Copyright 2023 Google LLC, Apache-2.0. All 19 original cells map in order through `frozen_source_cell`.

The native circuit API replaces the source builder and implicit allocator. Graphviz is an optional drawing dependency. The eight-bit tutorial constructs and validates its 23-wire physical circuit. Classical evaluation covers the eight-bit domain. Complete quantum branch checks use a smaller register to bound memory. Primitive counts, independent logical models and physical wire capacity remain distinct.
